# 5 · The CISO tool: "I just saw a new NVD entry — do we patch it now?"
`ciso_tool.py` scores CVEs with the **deployment model** (gradient boosting, same frozen settings, refit on all labelled
data 2022-01 → 2026-08-20; never used for any reported number) from **day-0 information only**:
* NVD record parsed with the **same code as training** (CNA CVSS, CNA CWE, CNA vendor/product, description);
* exploitation wording in the description counts only if the record was fetched within 7 days of publication (otherwise it
  cannot be proven day-0 and is removed);
* vendor/product **KEV history** counts only KEV entries added **before** the CVE's publication, from the newest local
  KEV snapshot (the tool never downloads KEV).

**Output per CVE:** within-month **percentile** vs the July 2026 reference month (primary), **indicative probability**
(Platt on time-ordered out-of-fold scores) and how many times the 0.31% base rate it is, **decision** PATCH NOW / MONITOR at
the **4.2%-of-monthly-CVEs budget**, and the **top 3 SHAP reasons** in plain words. Single CVEs are shown as a readable card,
a day's feed as a ranked table. CNA identifiers (often UUIDs) are shown as organisation names from `data/nvd_sources.csv`
(NVD source API, keyless; display only). NVD is read without an API key (the tool never reads `.env`); responses are cached in
`data/nvd_tool_cache/`, so this notebook re-runs offline. The same logic is available as a local web app with a 'Past CVEs' tab (frozen model's day-0 verdicts on the test period): `streamlit run app.py`.

Notebooks: [1 data](1_data.ipynb) · [2 train & validate](2_train_validate.ipynb) · [3 test](3_test.ipynb) · [4 results for the CISO](4_results_ciso.ipynb) · [5 CISO tool](5_ciso_tool.ipynb) · detailed appendices in `appendix/`.

In [1]:
import os, sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache" / "mpl")); os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
warnings.filterwarnings("ignore")
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
import ciso_tool as ct
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70); pd.set_option("display.max_columns", 20)
tool = ct.Tool(); print(tool.describe())
COLS = ["rank", "cve_id", "vendor", "product", "cna", "cna_cvss", "percentile", "probability_%", "decision", "reason_1", "reason_2", "reason_3"]

deploy model: 196,403 labelled CVEs (2022-01-01 -> 2026-08-20), sha256 627441c1e843 | KEV snapshot kev_20261120.csv | reference month 2026-07 (9,771 CVEs) | PATCH NOW = top 4.22% (score >= 0.6943)


## 5.1 Daily feed: everything NVD published on one day, ranked
Demo day **2026-09-08** (Microsoft Patch Tuesday), after the 2026-08-20 label cutoff, so none of these CVEs were in the
deployment model's training data. Top 20 shown with reasons.

In [2]:
DAY = "2026-09-08"
feed = tool.score(ct.fetch_day(DAY), explain_top=20)
n_now = (feed["decision"] == "PATCH NOW").sum()
print(f"{DAY}: {len(feed):,} CVEs published; PATCH NOW: {n_now} ({100 * n_now / len(feed):.1f}%) - the 4.2% budget is a MONTHLY share, so busy days such as Patch Tuesday take more")
display(feed.head(20)[COLS].round(1))
notes = feed[feed["notes"] != ""]; print(f"CVEs with a day-0 note (wording removed / KEV snapshot older than publication): {len(notes)}")

2026-09-08: 1,559 CVEs published; PATCH NOW: 182 (11.7%) - the 4.2% budget is a MONTHLY share, so busy days such as Patch Tuesday take more


,rank,cve_id,vendor,product,cna,cna_cvss,percentile,probability_%,decision,reason_1,reason_2,reason_3
0,1,CVE-2026-12744,Ivanti,Neurons for ITSM,ivanti,9.8,100.0,9.7,PATCH NOW,CNA CVSS score 9.8 -> strongly raises risk,published by CNA ivanti -> strongly raises risk,vendor's last KEV entry 89 days before publication -> moderately raises risk
1,2,CVE-2026-12745,Ivanti,Neurons for ITSM,ivanti,9.8,100.0,9.7,PATCH NOW,CNA CVSS score 9.8 -> strongly raises risk,published by CNA ivanti -> strongly raises risk,vendor's last KEV entry 89 days before publication -> moderately raises risk
2,3,CVE-2026-83527,Ivanti,Sentry,ivanti,8.1,100.0,8.3,PATCH NOW,published by CNA ivanti -> strongly raises risk,vendor's last KEV entry 89 days before publication -> strongly raises risk,vendor 'ivanti' -> moderately raises risk
3,4,CVE-2026-82004,Adobe,Adobe Campaign Classic,Adobe Systems Incorporated,10.0,99.9,6.6,PATCH NOW,CNA CVSS score 10.0 -> strongly raises risk,published by CNA Adobe Systems Incorporated -> moderately raises risk,vendor's last KEV entry 63 days before publication -> moderately raises risk
4,5,CVE-2026-26084,Fortinet,FortiSandbox,"Fortinet, Inc.",9.9,99.8,5.7,PATCH NOW,CNA CVSS score 9.9 -> strongly raises risk,vendor's last KEV entry 43 days before publication -> moderately raises risk,"published by CNA Fortinet, Inc. -> moderately raises risk"
5,6,CVE-2026-77089,Commvault,Commvault Cloud,Commvault,9.3,99.7,5.4,PATCH NOW,CNA CVSS score 9.3 -> strongly raises risk,published by CNA Commvault -> strongly raises risk,vendor's last KEV entry 494 days before publication -> moderately raises risk
6,7,CVE-2026-81376,Microsoft,Visual Studio Code,Microsoft Corporation,9.6,99.7,5.0,PATCH NOW,CNA CVSS score 9.6 -> strongly raises risk,weakness CWE-693 -> strongly raises risk,vendor's last KEV entry 13 days before publication -> moderately raises risk
7,8,CVE-2026-28659,Google,Android XR,Android (associated with Google Inc. or Open Handset Alliance),10.0,99.7,5.0,PATCH NOW,CNA CVSS score 10.0 -> strongly raises risk,published by CNA Android (associated with Google Inc. or Open Handset Alliance) -> moderately raises risk,vendor's last KEV entry 4 days before publication -> moderately raises risk
8,9,CVE-2026-49883,Google,Android Wear,Android (associated with Google Inc. or Open Handset Alliance),10.0,99.7,5.0,PATCH NOW,CNA CVSS score 10.0 -> strongly raises risk,vendor's last KEV entry 4 days before publication -> moderately raises risk,published by CNA Android (associated with Google Inc. or Open Handset Alliance) -> moderately raises risk
9,10,CVE-2026-69595,Microsoft,Windows Server 2012,Microsoft Corporation,9.8,99.6,4.5,PATCH NOW,CNA CVSS score 9.8 -> strongly raises risk,vendor's last KEV entry 13 days before publication -> strongly raises risk,published by CNA Microsoft Corporation -> moderately raises risk


CVEs with a day-0 note (wording removed / KEV snapshot older than publication): 0


### Anecdotal: which of these are in CISA KEV now?
> ⚠️ **Anecdotal, not a metric.** One day, a few CVEs, a KEV snapshot taken ~10 weeks later (2026-11-20, so the 90-day
> window is not complete). Shown only to illustrate the tool; the evaluated performance is in notebook 3.

In [3]:
kev = pd.read_csv(sorted((ROOT / "data" / "raw").glob("kev_*.csv"))[-1], usecols=["cveID", "dateAdded"])
f = feed.merge(kev.rename(columns={"cveID": "cve_id"}), on="cve_id", how="left")
in_kev = f[f["dateAdded"].notna()]
print(f"{len(in_kev)} of the {len(f):,} CVEs published on {DAY} are in the 2026-11-20 KEV snapshot:")
display(in_kev[["rank", "cve_id", "vendor", "product", "cna_cvss", "percentile", "decision", "dateAdded"]].round(1))
print(f"top 20 of the day that are in KEV: {f.head(20)['dateAdded'].notna().sum()} (anecdotal)")

3 of the 1,559 CVEs published on 2026-09-08 are in the 2026-11-20 KEV snapshot:


,rank,cve_id,vendor,product,cna_cvss,percentile,decision,dateAdded
28,29,CVE-2026-84869,ConnectWise,ScreenConnect,9.9,99.2,PATCH NOW,2026-09-11
103,104,CVE-2026-81963,Microsoft,Windows 11 version 23H2,7.8,97.4,PATCH NOW,2026-09-08
630,631,CVE-2026-85880,Microsoft,Windows 10 Version 1607,7.8,83.3,MONITOR,2026-09-08


top 20 of the day that are in KEV: 0 (anecdotal)


## 5.2 One CVE by ID (fetched from NVD)

In [4]:
one_id = feed.iloc[0]["cve_id"]
one = tool.score(ct.fetch_cve(one_id))
print(ct.card(one.iloc[0], tool))

  CVE-2026-12744   published 2026-09-08   CNA: ivanti
  Ivanti / Neurons for ITSM
  CVSS (CNA): 9.8  (CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H)
  DECISION:     PATCH NOW   (top 4.2% of a month's CVEs)
  Percentile:   100.0  (in the top 0.1% of the 9,771 CVEs of July 2026)
  Probability:  9.7% indicative chance of a CISA KEV listing within 90 days
                = 31.2x the base rate (0.31% of all CVEs)
  Why:
    1. CNA CVSS score 9.8 -> strongly raises risk
    2. published by CNA ivanti -> strongly raises risk
    3. vendor's last KEV entry 89 days before publication -> moderately raises risk


## 5.3 One CVE typed in by hand
For a CVE seen in a vendor advisory before NVD has it: vendor, CNA, CVSS vector (v3.x scored automatically), CWE and the
description. Example: a hypothetical unauthenticated remote code execution in a small vendor's product, vs the same bug in
a vendor with a long KEV history.

In [5]:
desc = "A stack-based buffer overflow in the web management interface allows an unauthenticated remote attacker to execute arbitrary code via a crafted HTTP request."
vec = "CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H"
manual = pd.concat([ct.manual_cve("Acme Widgets", "security@acme.example", desc, vec, cwe="CWE-121", product="WidgetOS", published="2026-11-20", cve_id="MANUAL-ACME"),
                    ct.manual_cve("Ivanti", "3c1d8aa1-5a33-4ea4-8992-aadd6440af75", desc, vec, cwe="CWE-121", product="Connect Secure", published="2026-11-20", cve_id="MANUAL-IVANTI")])
for _, row in tool.score(manual).iterrows():
    print(ct.card(row, tool))

  MANUAL-IVANTI   published 2026-11-20   CNA: ivanti
  Ivanti / Connect Secure
  CVSS (CNA): 9.8  (CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H)
  DECISION:     PATCH NOW   (top 4.2% of a month's CVEs)
  Percentile:   100.0  (in the top 0.1% of the 9,771 CVEs of July 2026)
  Probability:  10.0% indicative chance of a CISA KEV listing within 90 days
                = 32.1x the base rate (0.31% of all CVEs)
  Why:
    1. CNA CVSS score 9.8 -> strongly raises risk
    2. published by CNA ivanti -> strongly raises risk
    3. vendor's last KEV entry 21 days before publication -> moderately raises risk
  MANUAL-ACME   published 2026-11-20   CNA: security@acme.example
  Acme Widgets / WidgetOS
  CVSS (CNA): 9.8  (CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H)
  DECISION:     PATCH NOW   (top 4.2% of a month's CVEs)
  Percentile:   99.2  (scores higher than 99.2% of the 9,771 CVEs of July 2026)
  Probability:  3.1% indicative chance of a CISA KEV listing within 90 days
                = 9.9x t

**Takeaway**: The tool turns the evaluated model into a daily decision: each NVD entry gets a percentile against a full reference month, an indicative probability, a PATCH NOW / MONITOR decision at a volume-scaled budget, and the three facts that drove the score. On the demo day (1,559 CVEs, Patch Tuesday) 182 were PATCH NOW — more than 4.2% because the budget is a *monthly* share and Patch Tuesday is the busiest day. In the hand-typed example the same unauthenticated RCE description (CVSS 9.8) is PATCH NOW for both vendors, but the indicative probability is **10.0% for Ivanti vs 3.1% for an unknown vendor**: **who** is affected (KEV history, publisher) matters as much as severity, as in the SHAP analysis — which is why the model must be retrained as attacker attention moves (notebook 4).